# Определение ориентации текста

Ноутбук создаёт `submission.csv` с вероятностью поворота текста на 180° для 20 000 кропов. Используется локальная модель PaddleOCR `PP-LCNet_x1_0_textline_ori` на CPU. Каждый кроп проверяется в исходном положении и после поворота; в конце выводится время работы.

Запускайте из корня проекта с `models/` и `test_data/` (или `test.zip`). Другие пути можно задать через `AVITO_PROJECT_DIR` и `AVITO_TEST_PATH`. Зависимости перечислены в `requirements.txt`.


In [ ]:
from __future__ import annotations

import csv
import io
import math
import os
import random
import time
import zipfile
from pathlib import Path

import sys
# Torch здесь не нужен; отключаем его импорт в PaddleX, если локальная установка Torch повреждена.
sys.modules['torch'] = None


import numpy as np
import paddle
from PIL import Image
from paddleocr import TextLineOrientationClassification

# Пути можно переопределить, если ноутбук запускается из другой папки.
PROJECT_DIR = Path(os.environ.get("AVITO_PROJECT_DIR", Path.cwd())).expanduser().resolve()
MODEL_DIR = PROJECT_DIR / "models" / "PP-LCNet_x1_0_textline_ori_infer"
OUTPUT_DIR = Path(os.environ.get("AVITO_OUTPUT_DIR", PROJECT_DIR)).expanduser().resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV = OUTPUT_DIR / "submission.csv"
DEVICE = "cpu"
# В одной пачке будет 16 кропов и 16 их перевёрнутых копий.
BATCH_PAIRS = 16
SEED = 20260925

# Случайных аугментаций здесь нет, но seed фиксирую для воспроизводимости.
random.seed(SEED)
np.random.seed(SEED)
paddle.seed(SEED)

if not MODEL_DIR.is_dir():
    raise FileNotFoundError(f"Не найдены веса модели: {MODEL_DIR}")
print(f"Папка проекта: {PROJECT_DIR}")
print(f"Устройство: {DEVICE}")

In [ ]:
EXPECTED_ROWS = 20_000

def find_test_data(project_dir: Path) -> Path:
    # Сначала беру явно заданный путь, иначе ищу папку или архив рядом с проектом.
    explicit = os.environ.get("AVITO_TEST_PATH")
    if explicit:
        path = Path(explicit).expanduser().resolve()
        if not path.exists():
            raise FileNotFoundError(path)
        return path
    for path in (project_dir / "test_data", project_dir / "test.zip"):
        if path.exists():
            return path
    raise FileNotFoundError("Укажите AVITO_TEST_PATH или положите test_data/test.zip рядом с ноутбуком")

class TestData:
    """Читает изображения в порядке файла sample_submission.csv."""

    def __init__(self, path: Path):
        self.archive = None
        self.image_dir = None
        if path.is_file():
            self.archive = zipfile.ZipFile(path)
            sample = self.archive.read("sample_submission.csv")
        elif path.is_dir():
            if (path / "test" / "images").is_dir():
                self.image_dir = path / "test" / "images"
                sample_path = path / "sample_submission.csv"
            elif (path / "images").is_dir():
                self.image_dir = path / "images"
                sample_path = path / "sample_submission.csv"
                if not sample_path.is_file():
                    sample_path = path.parent / "sample_submission.csv"
            else:
                raise FileNotFoundError(f"Не найдена папка test/images в {path}")
            sample = sample_path.read_bytes()
        else:
            raise FileNotFoundError(path)
        # Порядок строк беру из образца отправки и сохраняю до самого конца.
        reader = csv.DictReader(io.StringIO(sample.decode("utf-8-sig")))
        if reader.fieldnames != ["image_id", "p_180"]:
            raise ValueError(f"Неверные столбцы в образце: {reader.fieldnames}")
        self.image_ids = [row["image_id"] for row in reader]
        if len(self.image_ids) != EXPECTED_ROWS or len(set(self.image_ids)) != EXPECTED_ROWS:
            raise ValueError("Ожидалось 20 000 уникальных image_id")

    def __enter__(self):
        return self

    def __exit__(self, *_):
        if self.archive is not None:
            self.archive.close()

    def read_image(self, image_id: str) -> bytes:
        name = f"{image_id}.png"
        if self.archive is not None:
            return self.archive.read(f"test/images/{name}")
        return (self.image_dir / name).read_bytes()

    def check_images(self) -> None:
        if self.archive is not None:
            available = set(self.archive.namelist())
            missing = [name for name in self.image_ids if f"test/images/{name}.png" not in available]
        else:
            missing = [name for name in self.image_ids if not (self.image_dir / f"{name}.png").is_file()]
        if missing:
            raise FileNotFoundError(f"Нет тестового изображения: {missing[0]}")

In [ ]:
PROBABILITY_LIMIT = 1e-6

def probability_180(result) -> float:
    label = result["label_names"][0]
    score = float(result["scores"][0])
    if label not in ("0_degree", "180_degree") or not math.isfinite(score):
        raise ValueError(f"Неожиданный ответ классификатора: {label}, {score}")
    if not 0 <= score <= 1:
        raise ValueError(f"Недопустимая уверенность классификатора: {score}")
    # Paddle возвращает уверенность выбранного класса, а нам нужна именно P(180°).
    return score if label == "180_degree" else 1 - score

def logit(value: float) -> float:
    # Для точных 0 и 1 логит не определён, поэтому слегка ограничиваю вероятность.
    value = min(1 - PROBABILITY_LIMIT, max(PROBABILITY_LIMIT, value))
    return math.log(value / (1 - value))

def sigmoid(value: float) -> float:
    if value >= 0:
        return 1 / (1 + math.exp(-value))
    exp_value = math.exp(value)
    return exp_value / (1 + exp_value)

def baseline_probability(original: float, rotated: float) -> float:
    # На перевёрнутом кропе классы меняются местами; усредняю два ответа в логитах.
    return sigmoid((logit(original) - logit(rotated)) / 2)

In [ ]:
total_start = time.perf_counter()
with TestData(find_test_data(PROJECT_DIR)) as test:
    test.check_images()

    start = time.perf_counter()
    model = TextLineOrientationClassification(
        model_name="PP-LCNet_x1_0_textline_ori",
        model_dir=str(MODEL_DIR),
        device=DEVICE,
    )
    model_load_seconds = time.perf_counter() - start

    image_seconds = 0.0
    inference_seconds = 0.0
    rows = []
    for start_index in range(0, len(test.image_ids), BATCH_PAIRS):
        ids = test.image_ids[start_index:start_index + BATCH_PAIRS]

        start = time.perf_counter()
        # Складываю изображения парами: сначала исходное, затем поворот на 180°.
        images = []
        for image_id in ids:
            with Image.open(io.BytesIO(test.read_image(image_id))) as source:
                image = source.convert("RGB")
            images.append(np.asarray(image))
            images.append(np.asarray(image.transpose(Image.Transpose.ROTATE_180)))
        image_seconds += time.perf_counter() - start

        start = time.perf_counter()
        # Всю пачку отдаю модели одним вызовом.
        predictions = model.predict(images, batch_size=2 * BATCH_PAIRS)
        inference_seconds += time.perf_counter() - start
        if len(predictions) != len(images):
            raise RuntimeError("Число ответов классификатора не совпадает с числом изображений")

        # Соседние ответы модели относятся к двум поворотам одного кропа.
        for position, image_id in enumerate(ids):
            p0 = probability_180(predictions[2 * position])
            p180 = probability_180(predictions[2 * position + 1])
            rows.append((image_id, baseline_probability(p0, p180)))

        done = start_index + len(ids)
        if done % 1024 == 0 or done == len(test.image_ids):
            print(f"Обработано {done}/{len(test.image_ids)}", flush=True)

    # Перед записью ещё раз сверяю порядок с образцом отправки.
    if [image_id for image_id, _ in rows] != test.image_ids:
        raise ValueError("Порядок image_id отличается от sample_submission.csv")

def write_submission(path: Path, predictions) -> None:
    # Сначала пишу временный файл: при сбое старый submission.csv не повредится.
    temporary = path.with_suffix(".csv.tmp")
    with temporary.open("w", encoding="utf-8", newline="") as file:
        writer = csv.writer(file, lineterminator="\n")
        writer.writerow(("image_id", "p_180"))
        writer.writerows((image_id, f"{p:.8f}") for image_id, p in predictions)
    os.replace(temporary, path)


start = time.perf_counter()
write_submission(OUTPUT_CSV, rows)
csv_seconds = time.perf_counter() - start
total_seconds = time.perf_counter() - total_start

print(f"Сохранено {len(rows)} прогнозов: {OUTPUT_CSV}")
print(f"Загрузка модели: {model_load_seconds:.2f} s")
print(f"Чтение и поворот: {image_seconds:.2f} s")
print(f"Работа модели: {inference_seconds:.2f} s")
print(f"Запись CSV: {csv_seconds:.2f} s")
print(f"Всего: {total_seconds:.2f} s ({total_seconds / 60:.2f} min)")
print(f"Скорость: {len(rows) / total_seconds:.1f} кропов/с")